# Stage 0: tests

**What this stage does:** checks that our code does the right thing **before** we let it loose on the real
data.

Each test builds a tiny made-up match where we worked out the right answer by hand, and compares the code's
answer with ours **exactly**. The tests cover the tricky cricket situations (strike changes, run-outs at the
non-striker's end, extras, two wickets on one ball, a batter coming back after retiring hurt, penalty runs),
the match filters, player IDs, how the networks are built, and the network measures.

**Stop rule:** the last cell stops with an error if any test fails. `run_pipeline.sh` runs this notebook first,
so nothing else runs if the code is broken.

In [1]:
%run ./library/lib_00_settings.ipynb
%run ./library/lib_01_data_loading.ipynb
%run ./library/lib_02_partnership_extraction.ipynb
%run ./library/lib_04_network_building.ipynb
%run ./library/lib_05_network_measures.ipynb

import math
import tempfile   # some tests write a small file; a temporary folder keeps the project folder clean

## Helpers for building made-up matches

`make_ball` builds one ball, `make_innings` puts balls into overs of six, and `make_match` wraps the innings
into something that looks like a Cricsheet file. `TEST_REGISTRY` plays the role of the player registry. We
added two traps on purpose: "A One" and "A. One" are the **same** player spelled two ways (same ID), while
"A One Jr" is a **different** player with a similar name.

In [2]:
TEST_REGISTRY = {"A One": "id_a", "B Two": "id_b", "C Three": "id_c", "D Four": "id_d", "E Five": "id_e",
                 "A. One": "id_a", "A One Jr": "id_a2", "R Roster": "id_r"}
A, B, C, D = "A One", "B Two", "C Three", "D Four"

def make_ball(batter, non_striker, total_runs=0, batter_runs=None, extras=0, wickets=None, **extra_fields):
    """One ball. If we don't say otherwise, the runs off the bat are the total minus the extras."""
    delivery = {"batter": batter, "non_striker": non_striker, "bowler": "X Bowler",
                "runs": {"batter": total_runs - extras if batter_runs is None else batter_runs,
                         "extras": extras, "total": total_runs}}
    if wickets:
        delivery["wickets"] = [{"player_out": player, "kind": kind} for player, kind in wickets]
    delivery.update(extra_fields)
    return delivery

def make_innings(deliveries, team="England", balls_per_over=6, **extra_fields):
    """Split a list of balls into overs and wrap them in an innings for the given team."""
    overs = [{"over": start // balls_per_over, "deliveries": deliveries[start:start + balls_per_over]}
             for start in range(0, len(deliveries), balls_per_over)]
    return {"team": team, "overs": overs, **extra_fields}

def make_match(innings_list, match_day="2022-06-01", match_type="ODI", gender="male",
               team_type="international", teams=("England", "Australia"), players=None):
    """A complete made-up match file. The defaults describe a normal England ODI in 2022."""
    return {"meta": {"data_version": "1.3.0", "created": "2022-06-02", "revision": 1},
            "info": {"dates": [match_day], "gender": gender, "team_type": team_type, "match_type": match_type,
                     "teams": list(teams), "outcome": {}, "players": players or {},
                     "registry": {"people": dict(TEST_REGISTRY)}},
            "innings": innings_list}

def extract(deliveries, **extra_fields):
    """Run the partnership extractor on one made-up England innings."""
    return extract_innings_partnerships(make_innings(deliveries, **extra_fields), TEST_REGISTRY, "m1", 1)

def pairs_and_runs(innings_result):
    """The result as a list of (player 1, player 2, runs), which is easy to compare with our hand answer."""
    return [(obs.player_1_id, obs.player_2_id, obs.partnership_runs) for obs in innings_result.observations]

def exclusion_reason_for(**match_settings):
    """The filter decision for an empty made-up match with the given settings."""
    return find_exclusion_reason(describe_match(make_match([], **match_settings), "1.json"))

def build_tables(matches):
    """Run the whole chain (filters, extractor, node/edge tables) on a few made-up matches.

    matches is a list of (match_id, match). Returns the partnership, node, edge and player tables.
    """
    observation_rows = []
    for match_id, match_data in matches:
        match_record = describe_match(match_data, f"{match_id}.json")
        assert find_exclusion_reason(match_record) == ""
        observation_rows += extract_match_partnerships(match_data, match_record)[0]
    observations = pd.DataFrame(observation_rows, columns=OBSERVATION_COLUMNS)
    player_table = build_player_table(observations)
    return (observations, build_node_table(observations, player_table),
            build_edge_table(observations, player_table), player_table)

def close_to(value, expected):
    """Compare decimals safely (0.1 + 0.2 is not exactly 0.3 on a computer)."""
    return math.isclose(value, expected, rel_tol=1e-9, abs_tol=1e-12)

## Partnership tests

The numbers in the test names are only there to keep them in order.

In [3]:
def test_01_simple_partnership():
    result = extract([make_ball(A, B, 1), make_ball(B, A, 4), make_ball(B, A, 0), make_ball(B, A, 2)])
    assert pairs_and_runs(result) == [("id_a", "id_b", 7)]
    only = result.observations[0]
    assert (only.deliveries, only.start_delivery_id, only.end_delivery_id) == (4, "1.0.1", "1.0.4")
    assert only.boundary_reason == "innings_end" and result.warnings == []

def test_02_strike_rotation_keeps_partnership():
    result = extract([make_ball(A, B, 1), make_ball(B, A, 1), make_ball(A, B, 3), make_ball(B, A, 1), make_ball(A, B, 0)])
    assert pairs_and_runs(result) == [("id_a", "id_b", 6)]

def test_03_wicket_ball_runs_go_to_old_pair():
    result = extract([make_ball(A, B, 4), make_ball(A, B, 1, wickets=[(A, "caught")]),
                      make_ball(C, B, 2), make_ball(B, C, 3)])
    assert pairs_and_runs(result) == [("id_a", "id_b", 5), ("id_b", "id_c", 5)]
    assert result.observations[0].boundary_reason == "dismissal:caught"
    assert result.observations[0].end_delivery_id == "1.0.2"
    assert result.observations[1].start_delivery_id == "1.0.3"
    assert result.observations[1].partnership_seq == 2 and result.warnings == []

def test_04_non_striker_run_out_closes_partnership():
    result = extract([make_ball(A, B, 2), make_ball(A, B, 1, wickets=[(B, "run out")]), make_ball(A, C, 6)])
    assert pairs_and_runs(result) == [("id_a", "id_b", 3), ("id_a", "id_c", 6)]
    assert result.observations[0].boundary_reason == "dismissal:run out"

def test_05_extras_counted_once():
    # A wide (total 1) and a no-ball hit for four (total 5 = 4 off the bat + 1 extra).
    # If extras were added on top of runs.total we would get 10 instead of 8.
    result = extract([make_ball(A, B, 1, extras=1, extras_detail={"wides": 1}),
                      make_ball(A, B, 5, batter_runs=4, extras=1), make_ball(A, B, 2)])
    only = result.observations[0]
    assert only.partnership_runs == 8
    assert (only.batter_runs, only.extras_runs) == (6, 2)
    assert result.runs_from_deliveries == 8

def test_06_two_wickets_on_one_ball_no_fake_partnership():
    result = extract([make_ball(A, B, 3), make_ball(A, B, 0, wickets=[(A, "retired hurt"), (B, "run out")]),
                      make_ball(C, D, 4)])
    assert pairs_and_runs(result) == [("id_a", "id_b", 3), ("id_c", "id_d", 4)]
    assert result.observations[0].boundary_reason == "dismissal:retired hurt+run out"
    assert result.wicket_records == 2 and result.warnings == []

def test_07_innings_end_closes_last_partnership_once():
    result = extract([make_ball(A, B, 1), make_ball(A, B, 0, wickets=[(A, "bowled")]), make_ball(C, B, 2)])
    assert [obs.boundary_reason for obs in result.observations] == ["dismissal:bowled", "innings_end"]
    ends_on_wicket = extract([make_ball(A, B, 1), make_ball(A, B, 0, wickets=[(A, "bowled")])])
    assert len(ends_on_wicket.observations) == 1
    assert ends_on_wicket.observations[0].boundary_reason == "dismissal:bowled"

def test_08_reunited_pair_gives_two_observations():
    result = extract([make_ball(A, B, 5), make_ball(A, B, 0, wickets=[(A, "retired hurt")]),
                      make_ball(C, B, 2), make_ball(C, B, 0, wickets=[(C, "lbw")]), make_ball(A, B, 7)])
    assert pairs_and_runs(result) == [("id_a", "id_b", 5), ("id_b", "id_c", 2), ("id_a", "id_b", 7)]
    assert result.observations[0].boundary_reason == "retirement:retired hurt"
    assert result.warnings == []

def test_15_innings_penalty_not_given_to_a_pair():
    result = extract([make_ball(A, B, 2), make_ball(A, B, 1)], penalty_runs={"pre": 5})
    assert pairs_and_runs(result) == [("id_a", "id_b", 3)]
    assert result.runs_from_deliveries == 3

def test_unexplained_pair_change_is_logged():
    result = extract([make_ball(A, B, 2), make_ball(C, B, 1)])
    assert pairs_and_runs(result) == [("id_a", "id_b", 2), ("id_b", "id_c", 1)]
    assert result.observations[0].boundary_reason == "unexplained_pair_change"
    assert result.observations[0].quality_status == "warning"
    assert [warning["code"] for warning in result.warnings] == ["unexplained_pair_change"]

def test_recorded_batter_replacement_is_a_valid_boundary():
    replacement = {"role": [{"in": C, "out": A, "reason": "injury", "role": "batter"}]}
    result = extract([make_ball(A, B, 2), make_ball(C, B, 1, replacements=replacement)])
    assert result.observations[0].boundary_reason == "replacement" and result.warnings == []

def test_dismissed_batter_seen_again_is_flagged():
    result = extract([make_ball(A, B, 0, wickets=[(A, "bowled")]), make_ball(A, B, 1)])
    assert [warning["code"] for warning in result.warnings] == ["player_out_still_at_crease"]

## Filter, player ID and network-building tests

In [4]:
def test_11_date_boundaries():
    assert exclusion_reason_for(match_day="2020-01-01") == ""
    assert exclusion_reason_for(match_day="2026-10-29") == ""
    assert exclusion_reason_for(match_day="2019-12-31") == "before_window"
    assert exclusion_reason_for(match_day="2026-10-30") == "after_window"

def test_12_gender_team_and_format_filters():
    assert exclusion_reason_for(gender="female") == "gender_not_male"
    assert exclusion_reason_for(team_type="club", match_type="T20") == "not_international"
    assert exclusion_reason_for(teams=("Australia", "India")) == "england_not_playing"
    assert exclusion_reason_for(match_type="MDM") == "format_not_target"
    assert exclusion_reason_for(match_type="T20") == "" and exclusion_reason_for(match_type="Test") == ""
    assert describe_match(make_match([], match_type="T20"), "1.json")["format"] == "T20I"

def test_09_same_pair_in_two_matches_adds_up():
    first_match = make_match([make_innings([make_ball(A, B, 10), make_ball(A, B, 5)])], match_day="2021-05-01")
    second_match = make_match([make_innings([make_ball(B, A, 20)])], match_day="2023-07-09")
    observations, node_table, edge_table, _ = build_tables([("m1", first_match), ("m2", second_match)])
    assert len(edge_table) == 1
    edge = edge_table.iloc[0]
    assert (edge.weight, edge.partnership_count, edge.total_matches, edge.total_innings) == (35, 2, 2, 2)
    assert (edge.first_date, edge.last_date) == (date(2021, 5, 1), date(2023, 7, 9))
    assert (edge.min_partnership_runs, edge.max_partnership_runs, edge.mean_partnership_runs) == (15, 20, 17.5)
    assert all(passed for _, passed, _ in aggregation_checks(observations, node_table, edge_table, "ODI"))

def test_08b_reunited_pair_is_one_edge_with_two_partnerships():
    match_data = make_match([make_innings([
        make_ball(A, B, 5), make_ball(A, B, 0, wickets=[(A, "retired hurt")]),
        make_ball(C, B, 2), make_ball(C, B, 0, wickets=[(C, "lbw")]), make_ball(A, B, 7)])])
    _, node_table, edge_table, _ = build_tables([("m1", match_data)])
    pair_ab = edge_table[(edge_table.player_1_id == "id_a") & (edge_table.player_2_id == "id_b")].iloc[0]
    assert (pair_ab.weight, pair_ab.partnership_count, pair_ab.total_matches, pair_ab.total_innings) == (12, 2, 1, 1)
    player_b = node_table[node_table.player_id == "id_b"].iloc[0]
    assert (player_b.matches_batted, player_b.innings_batted,
            player_b.partnership_observations, player_b.distinct_partners) == (1, 1, 3, 2)
    entry_point = node_table.set_index("player_id")["median_entry_partnership"]
    assert (entry_point["id_a"], entry_point["id_b"], entry_point["id_c"]) == (1, 1, 2)

def test_10_same_pair_in_two_formats_stays_separate():
    test_match = make_match([make_innings([make_ball(A, B, 10)])], match_type="Test")
    t20_match = make_match([make_innings([make_ball(A, B, 3)])], match_type="T20")
    _, node_table, edge_table, _ = build_tables([("m1", test_match), ("m2", t20_match)])
    assert sorted(edge_table["format"]) == ["T20I", "Test"]
    assert build_format_graph(node_table, edge_table, "Test")["id_a"]["id_b"]["weight"] == 10
    assert build_format_graph(node_table, edge_table, "T20I")["id_a"]["id_b"]["weight"] == 3
    assert build_format_graph(node_table, edge_table, "ODI").number_of_nodes() == 0

def test_13_player_identity_uses_ids_not_names():
    match_1 = make_match([make_innings([make_ball("A One", B, 4), make_ball("A One", B, 4)])])
    match_2 = make_match([make_innings([make_ball("A. One", B, 6)])], match_day="2022-06-02")
    match_3 = make_match([make_innings([make_ball("A One Jr", B, 1)])], match_day="2022-06-03")
    _, node_table, edge_table, player_table = build_tables([("m1", match_1), ("m2", match_2), ("m3", match_3)])
    assert sorted(node_table.player_id) == ["id_a", "id_a2", "id_b"]
    assert edge_table.set_index("pair_key").loc["id_a__id_b", "weight"] == 14
    players = player_table.set_index("player_id")
    assert players.loc["id_a", "name_variants"] == "A One|A. One"
    assert players.loc["id_a", "player_name"] == "A One"

def test_14_squad_only_player_is_not_a_node():
    match_data = make_match([make_innings([make_ball(A, B, 1)])], players={"England": [A, B, "R Roster"]})
    _, node_table, _, _ = build_tables([("m1", match_data)])
    assert "id_r" not in set(node_table.player_id) and len(node_table) == 2

def test_opponent_and_super_over_innings_are_ignored():
    match_data = make_match([make_innings([make_ball(A, B, 9)], team="Australia"),
                             make_innings([make_ball(A, B, 4)]),
                             make_innings([make_ball(A, C, 12)], super_over=True)])
    observation_rows, innings_rows, _ = extract_match_partnerships(match_data, describe_match(match_data, "m1.json"))
    assert [(row["innings_index"], row["partnership_runs"]) for row in observation_rows] == [(2, 4)]
    assert [(row["innings_index"], row["qualifying"]) for row in innings_rows] == [(2, True), (3, False)]

def test_15b_penalty_runs_recorded_separately():
    match_data = make_match([make_innings([make_ball(A, B, 2)], penalty_runs={"pre": 5})])
    observation_rows, innings_rows, _ = extract_match_partnerships(match_data, describe_match(match_data, "m1.json"))
    assert observation_rows[0]["partnership_runs"] == 2
    assert (innings_rows[0]["penalty_runs_pre"], innings_rows[0]["partnership_runs_sum"]) == (5, 2)

def test_16_graph_is_undirected_and_weighted():
    match_data = make_match([make_innings([make_ball(A, B, 5), make_ball(A, B, 0, wickets=[(A, "bowled")]),
                                           make_ball(C, B, 8)])])
    _, node_table, edge_table, _ = build_tables([("m1", match_data)])
    partnership_graph = build_format_graph(node_table, edge_table, "ODI")
    assert not partnership_graph.is_directed()
    assert (partnership_graph.number_of_nodes(), partnership_graph.number_of_edges()) == (3, 2)
    # The edge can be looked up from either end, and carries the runs as its weight.
    assert partnership_graph["id_b"]["id_a"]["weight"] == partnership_graph["id_a"]["id_b"]["weight"] == 5
    assert partnership_graph["id_b"]["id_c"]["weight"] == 8

def test_unreadable_file_is_logged_not_skipped():
    with tempfile.TemporaryDirectory() as temporary_folder:
        good_file, broken_file = Path(temporary_folder) / "1.json", Path(temporary_folder) / "2.json"
        good_file.write_text(json.dumps(make_match([])))
        broken_file.write_text("{not json")
        file_audit = audit_match_files([good_file, broken_file]).set_index("filename")
    assert bool(file_audit.loc["1.json", "included"]) is True
    assert file_audit.loc["2.json", "exclusion_reason"].startswith("unreadable")

## Network-measure tests

For these we use a tiny network where we can work out every answer on paper: a triangle a, b, c, plus a fourth
batter d who only batted with a.

In [5]:
def tiny_network() -> nx.Graph:
    """Triangle a-b-c plus d hanging off a. The weights are made up and very different on purpose."""
    partnership_graph = nx.Graph(format="ODI")
    for player in "abcd":
        partnership_graph.add_node(player, format="ODI", player_name=player.upper(), matches_batted=2,
                                   innings_batted=2, partnership_observations=3)
    partnership_graph.add_edge("a", "b", weight=100, partnership_count=2)
    partnership_graph.add_edge("a", "c", weight=10, partnership_count=1)
    partnership_graph.add_edge("b", "c", weight=1, partnership_count=1)
    partnership_graph.add_edge("a", "d", weight=50, partnership_count=1)
    return partnership_graph

def test_node_measure_values():
    measures = node_measures(tiny_network()).set_index("player_id")
    assert measures.loc["a", "degree"] == 3 and measures.loc["a", "strength"] == 160
    assert measures.loc["a", "strength_per_innings"] == 80 and measures.loc["a", "degree_centrality"] == 1.0
    # a lies on the shortest paths b-d and c-d: 2 of the 3 pairs it is not part of.
    assert close_to(measures.loc["a", "betweenness"], 2 / 3)
    assert measures.loc["b", "betweenness"] == 0
    # a is one step from everyone; d is 1 step from a and 2 from b and c, so 3 / (1 + 2 + 2).
    assert measures.loc["a", "closeness"] == 1.0 and close_to(measures.loc["d", "closeness"], 3 / 5)
    # a's partners b, c, d form 3 pairs, only b-c is tied: clustering 1/3. b's two partners are tied: 1.
    assert close_to(measures.loc["a", "clustering"], 1 / 3) and measures.loc["b", "clustering"] == 1.0
    # d has only one partner, so by convention its clustering is 0.
    assert measures.loc["d", "clustering"] == 0
    assert measures.loc["a", "core_number"] == 2 and measures.loc["d", "core_number"] == 1
    # Redundancy = 2t / d. For a: one tie among its 3 partners, so 2/3. For b: partners a and c are tied, 2/2 = 1.
    assert close_to(measures.loc["a", "redundancy"], 2 / 3)
    assert close_to(measures.loc["b", "redundancy"], 1.0)
    assert measures.loc["d", "redundancy"] == 0
    assert measures["eigenvector"].idxmax() == "a"
    assert measures.index[0] == "a"   # sorted by strength

def test_weights_do_not_change_path_measures():
    weighted, unweighted = tiny_network(), tiny_network()
    for _, _, attributes in unweighted.edges(data=True):
        attributes["weight"] = 1
    weighted_measures = node_measures(weighted).set_index("player_id")
    unweighted_measures = node_measures(unweighted).set_index("player_id")
    # These measures are supposed to ignore the runs, so changing all weights to 1 must not change them.
    for measure_name in ("betweenness", "closeness", "clustering", "core_number", "redundancy"):
        assert weighted_measures[measure_name].equals(unweighted_measures[measure_name].reindex(weighted_measures.index)), measure_name

def test_network_summary_values():
    partnership_graph = tiny_network()
    summary = network_summary(partnership_graph, node_measures(partnership_graph))
    assert (summary["nodes"], summary["edges"], summary["components"], summary["main_component_nodes"]) == (4, 4, 1, 4)
    assert close_to(summary["density"], 4 / 6)            # 4 ties out of 6 possible
    assert close_to(summary["transitivity"], 3 / 5)       # 1 triangle (x3) over 5 connected triples
    # Distances: a-b, a-c, a-d, b-c are 1 and b-d, c-d are 2, so the mean is 8/6.
    assert close_to(summary["mean_geodesic_distance"], 8 / 6)
    assert summary["diameter"] == 2 and summary["edge_weight_total"] == 161
    # Everyone can reach everyone: connectedness 1. Compactness = (4 x 1 + 2 x 1/2) / 6 = 5/6.
    assert summary["connectedness"] == 1.0 and close_to(summary["compactness"], 5 / 6)
    assert summary["component_ratio"] == 0
    assert close_to(summary["mean_redundancy"], (2 / 3 + 1 + 1 + 0) / 4)
    assert summary["max_core_k"] == 2 and summary["max_core_nodes"] == 3

def test_degree_centralisation_extremes():
    assert close_to(degree_centralisation(nx.star_graph(5)), 1.0)
    assert degree_centralisation(nx.cycle_graph(6)) == 0

def test_disconnected_network_is_handled():
    # Our real networks are all one piece, but the code must still behave if a network falls apart.
    partnership_graph = tiny_network()
    for player in "xy":
        partnership_graph.add_node(player, format="ODI", player_name=player, matches_batted=1,
                                   innings_batted=1, partnership_observations=1)
    partnership_graph.add_edge("x", "y", weight=5, partnership_count=1)
    measures = node_measures(partnership_graph).set_index("player_id")
    assert not measures.loc["x", "in_main_component"] and math.isnan(measures.loc["x", "closeness"])
    summary = network_summary(partnership_graph, measures.reset_index())
    assert (summary["components"], summary["main_component_nodes"]) == (2, 4)
    assert close_to(summary["component_ratio"], 1 / 5)            # (2 - 1) / (6 - 1)
    # Reachable ordered pairs: 4 x 3 in the big piece + 2 in the small one = 14 out of 6 x 5 = 30.
    assert close_to(summary["connectedness"], 14 / 30)

def test_louvain_is_repeatable_with_a_fixed_seed():
    karate_club = nx.karate_club_graph()
    assert louvain_communities(karate_club, seed=42) == louvain_communities(karate_club, seed=42)

## Run every test

We collect every function whose name starts with `test_`, run it, and list the results in a table.

In [6]:
test_functions = {name: function for name, function in list(globals().items())
                  if name.startswith("test_") and callable(function)}
test_results = []
for test_name, test_function in test_functions.items():
    try:
        test_function()
        test_results.append((test_name, "passed", ""))
    except Exception as error:
        test_results.append((test_name, "FAILED", f"{type(error).__name__}: {error}"))

test_result_table = pd.DataFrame(test_results, columns=["test", "result", "error"])
number_passed = int((test_result_table["result"] == "passed").sum())
print(f"{number_passed} of {len(test_result_table)} tests passed")
assert number_passed == len(test_result_table), test_result_table[test_result_table["result"] != "passed"].to_string()
test_result_table

29 of 29 tests passed


,test,result,error
0,test_01_simple_partnership,passed,
1,test_02_strike_rotation_keeps_partnership,passed,
2,test_03_wicket_ball_runs_go_to_old_pair,passed,
3,test_04_non_striker_run_out_closes_partnership,passed,
4,test_05_extras_counted_once,passed,
5,test_06_two_wickets_on_one_ball_no_fake_partne...,passed,
6,test_07_innings_end_closes_last_partnership_once,passed,
7,test_08_reunited_pair_gives_two_observations,passed,
8,test_15_innings_penalty_not_given_to_a_pair,passed,
9,test_unexplained_pair_change_is_logged,passed,
